In [ ]:
# This notebook mirrors notebooks/07/linear_dim_reduction.py (the Marimo
# deck that is actually rendered to the course site). The figures are
# static equivalents of the deck's interactive ones.
from IPython.display import display, Markdown
import numpy as np
import matplotlib.pyplot as plt


# Data transformation and dimensionality reduction (DTDR) I


## Introduction

- So far, the course has been problem oriented (mainly classification).
- However, an essential part of pattern recognition is data analysis.
- In this lecture, we will look at methods that compress data into a more compact representation through linear transformations.


## Analyzing data

- Real data often live in a high-dimensional space, but the information
  they carry may be concentrated on a much smaller subspace.
- Example below: two features that are almost perfectly correlated.
    - Little is lost by describing the cloud with a single direction.


In [ ]:
rng_corr = np.random.default_rng(0)
cov_corr = np.array([[1.0, 0.95], [0.95, 1.0]])
X_corr = rng_corr.multivariate_normal([0.0, 0.0], cov_corr, size=400)

fig_corr = plt.figure(figsize=(10, 5))
gs_corr = fig_corr.add_gridspec(
    2, 2, width_ratios=(4, 1), height_ratios=(1, 4), wspace=0.05, hspace=0.05
)
ax_corr = fig_corr.add_subplot(gs_corr[1, 0])
ax_corr_top = fig_corr.add_subplot(gs_corr[0, 0], sharex=ax_corr)
ax_corr_right = fig_corr.add_subplot(gs_corr[1, 1], sharey=ax_corr)

ax_corr.scatter(X_corr[:, 0], X_corr[:, 1], s=12, alpha=0.5)
ax_corr_top.hist(X_corr[:, 0], bins=30, color="tab:blue")
ax_corr_right.hist(X_corr[:, 1], bins=30, orientation="horizontal", color="tab:blue")
ax_corr.set_xlabel("$x_1$")
ax_corr.set_ylabel("$x_2$")
ax_corr_top.tick_params(labelbottom=False)
ax_corr_right.tick_params(labelleft=False)
ax_corr_top.set_title(r"Two highly correlated features ($\rho = 0.95$)")

display(fig_corr)
plt.close(fig_corr)


### Problems

- The data dimensionality may be very large.
    - Computationally demanding.
    - Curse of dimensionality
- Some parts of the data may not be discriminative / be redundant.


### Remedies

- May pick only parts of the data to use.
- May transform the data!


### Approaches to DTDR

- To better discriminate between classes (supervised).
- To remove redundancy (unsupervised).


## Fisher discriminant analysis (FDA)

- Transform (project) to 1D: $z = \mathbf{w}^T \mathbf{x}$
- Start with the two class case and $P(w_1)=P(w_2)$
- Fisher discriminant ratio (FDR) — large is good:

$$
\mathrm{FDR} = \frac{(\mu_1 - \mu_2)^2}{\sigma_1^2 + \sigma_2^2}
$$

where $\mu_i$ and $\sigma_i^2$ are the mean and variance of the
projected class-$i$ data.


### Scatter matrices

- Within class: $\boldsymbol{S}_w = \sum\limits_{i=1}^{M} P(w_i) \boldsymbol{\Sigma}_i$

- where $\boldsymbol{\Sigma}_i = \mathbb{E}\left[(\mathbf{x} - \boldsymbol{\mu}_i)(\mathbf{x} - \boldsymbol{\mu}_i)^T\right]$

---

- Between class: $\boldsymbol{S}_B = \sum\limits_{i=1}^{M} P(w_i) (\boldsymbol{\mu}_i - \boldsymbol{\mu})(\boldsymbol{\mu}_i - \boldsymbol{\mu})^T$

- where $\boldsymbol{\mu} = \sum\limits_{i=1}^{M} P(w_i) \boldsymbol{\mu}_i$ is the global mean.

- Both are $d \times d$ matrices: $\boldsymbol{S}_w$ measures spread *inside* classes, $\boldsymbol{S}_B$ spread *between* class means.


In [ ]:
from matplotlib.patches import Ellipse

# Two 2-D Gaussian classes reused by the Fisher-projection figure below.
# Equal priors, so the within/between scatter reduces to simple averages.
rng_fda = np.random.default_rng(0)
mu_fda_1 = np.array([-1.5, 0.5])
mu_fda_2 = np.array([1.5, -0.5])
cov_fda_1 = np.array([[1.0, 0.6], [0.6, 1.0]])
cov_fda_2 = np.array([[0.8, -0.4], [-0.4, 0.9]])
X_fda_1 = rng_fda.multivariate_normal(mu_fda_1, cov_fda_1, size=200)
X_fda_2 = rng_fda.multivariate_normal(mu_fda_2, cov_fda_2, size=200)

fig_scat, ax_scat = plt.subplots(figsize=(7, 6))
ax_scat.scatter(X_fda_1[:, 0], X_fda_1[:, 1], s=12, alpha=0.4, color="tab:blue", label="$w_1$")
ax_scat.scatter(X_fda_2[:, 0], X_fda_2[:, 1], s=12, alpha=0.4, color="tab:orange", label="$w_2$")
# 2-sigma covariance ellipses: the "within-class" scatter S_w.
for mu_i, cov_i, col_i in [
    (mu_fda_1, cov_fda_1, "tab:blue"),
    (mu_fda_2, cov_fda_2, "tab:orange"),
]:
    w_i, v_i = np.linalg.eigh(cov_i)
    order_i = np.argsort(w_i)[::-1]
    w_i, v_i = w_i[order_i], v_i[:, order_i]
    ang_i = np.degrees(np.arctan2(v_i[1, 0], v_i[0, 0]))
    ax_scat.add_patch(
        Ellipse(mu_i, *(4 * np.sqrt(w_i)), angle=ang_i, fill=False,
                edgecolor=col_i, lw=2, ls="--")
    )
    ax_scat.plot(*mu_i, marker="X", color=col_i, ms=14, mec="k")
mu_fda = 0.5 * (mu_fda_1 + mu_fda_2)
ax_scat.plot(*mu_fda, marker="*", color="k", ms=18, label=r"global mean $\mu$")
ax_scat.set_xlabel("$x_1$")
ax_scat.set_ylabel("$x_2$")
ax_scat.set_aspect("equal")
ax_scat.legend()
ax_scat.set_title("Within-class scatter $S_w$ and class means")

display(fig_scat)
plt.close(fig_scat)


### Remark

- Class separability in $\mathbf{x}$ can be measured by e.g.

$$\frac{\operatorname{trace}(\boldsymbol{S}_w)}{\operatorname{trace}(\boldsymbol{S}_B)}$$

- Here **small is good**: little within-class spread relative to the between-class spread.


### Fisher discriminant analysis

- Remember; want to learn a transformation into 1D $z = \mathbf{w}^T \mathbf{x} \;\Rightarrow\; \mu_z = \mathbf{w}^T \boldsymbol{\mu}$

- In the projected space, with equal priors $P(w_1)=P(w_2)=\tfrac{1}{2}$:

$$S_B = (\mu_1 - \mu_2)^2, \qquad \mu_i = \mathbf{w}^T \boldsymbol{\mu}_i$$

$$\sigma_i^2 = \mathbb{E}\left[(z - \mu_i)^2\right] = \mathbf{w}^T \boldsymbol{\Sigma}_i \mathbf{w}$$

$$S_w = \tfrac{1}{2}\sigma_1^2 + \tfrac{1}{2}\sigma_2^2$$

- Hence: Fisher discriminant ratio (FDR) — to be **maximized**:

$$\mathrm{FDR}(\mathbf{w}) = \frac{(\mu_1 - \mu_2)^2}{\sigma_1^2 + \sigma_2^2}$$


### FDR

- Want:

$$\arg\max_{\mathbf{w}} \frac{\mathbf{w}^T \boldsymbol{S}_B \mathbf{w}}{\mathbf{w}^T \boldsymbol{S}_w \mathbf{w}}$$

- At the solution (generalized eigenvalue problem):
  $\boldsymbol{S}_B \mathbf{w} = \lambda \boldsymbol{S}_w \mathbf{w}$
  $\;\Leftrightarrow\;$ $\boldsymbol{S}_w^{-1} \boldsymbol{S}_B \mathbf{w} = \lambda \mathbf{w}$


### Eigenvalue problems

- For a square matrix $\mathbf{A}$, an **eigenvector** $\mathbf{v} \neq \mathbf{0}$ and its **eigenvalue** $\lambda$ satisfy

$$
\mathbf{A} \mathbf{v} = \lambda \mathbf{v}
$$

- $\mathbf{A}$ stretches $\mathbf{v}$ by $\lambda$ without changing its direction.

---

- For the covariance matrix $\boldsymbol{\Sigma}$ (symmetric, positive semi-definite):

$$
\boldsymbol{\Sigma} \mathbf{e}_i = \lambda_i \mathbf{e}_i, \qquad \lambda_i \geq 0
$$

- The eigenvectors are orthogonal and the eigenvalue $\lambda_i$ is the **variance** of the data along $\mathbf{e}_i$.
- The largest eigenvalue gives the **direction of greatest variance** (the first principal component).


In [ ]:
from matplotlib.patches import Ellipse

rng_eig = np.random.default_rng(1)
cov_eig = np.array([[2.0, 1.4], [1.4, 1.2]])
X_eig = rng_eig.multivariate_normal([0.0, 0.0], cov_eig, size=400)
mu_eig = X_eig.mean(axis=0)
lam_eig, E_eig = np.linalg.eigh(cov_eig)
order_eig = np.argsort(lam_eig)[::-1]
lam_eig, E_eig = lam_eig[order_eig], E_eig[:, order_eig]

fig_eig, ax_eig = plt.subplots(figsize=(7, 6))
ax_eig.scatter(X_eig[:, 0], X_eig[:, 1], s=12, alpha=0.35, color="gray")
ang_eig = np.degrees(np.arctan2(E_eig[1, 0], E_eig[0, 0]))
for n_std_eig, alpha_eig in [(1, 0.5), (2, 0.2)]:
    ax_eig.add_patch(Ellipse(
        mu_eig, *(2 * n_std_eig * np.sqrt(lam_eig)), angle=ang_eig,
        fill=False, edgecolor="tab:red", lw=2, alpha=alpha_eig,
    ))
for i_eig in range(2):
    v_eig = E_eig[:, i_eig] * (2 * np.sqrt(lam_eig[i_eig]))
    ax_eig.annotate("", xy=mu_eig + v_eig, xytext=mu_eig,
                    arrowprops=dict(arrowstyle="->", lw=3, color=f"C{i_eig}"))
    ax_eig.text(*(mu_eig + v_eig * 1.15),
                f"$e_{{{i_eig + 1}}}$  $\\lambda_{{{i_eig + 1}}} = {lam_eig[i_eig]:.2f}$",
                color=f"C{i_eig}", fontsize=13)
ax_eig.set_xlabel("$x_1$")
ax_eig.set_ylabel("$x_2$")
ax_eig.set_aspect("equal")
ax_eig.set_title("Eigenvectors of $\\Sigma$: directions of greatest variance")

display(fig_eig)
plt.close(fig_eig)


## Solving the Fisher discriminant ratio

I. If $P(w_1) = P(w_2)$:
$$
\lambda \boldsymbol{S}_w \mathbf{w} = (\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2)(\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2)^T \mathbf{w}
$$
$\implies$ $\mathbf{w} \propto \boldsymbol{S}_w^{-1} (\boldsymbol{\mu}_1 - \boldsymbol{\mu}_2)$

---

II. If $P(w_1) \neq P(w_2)$:
$$
\mathbf{w} = \text{leading eigenvector of } \boldsymbol{S}_w^{-1} \boldsymbol{S}_B
$$


In [ ]:
# Fisher projection: a suboptimal direction vs the Fisher optimum
# w = S_w^{-1}(mu_1 - mu_2). The FDR is largest at the optimum.
def fdr_of(w, A, B):
    za, zb = A @ w, B @ w
    return (za.mean() - zb.mean()) ** 2 / (za.var() + zb.var())


S_w_fda = 0.5 * (cov_fda_1 + cov_fda_2)
w_opt_fda = np.linalg.solve(S_w_fda, mu_fda_1 - mu_fda_2)
w_opt_fda = w_opt_fda / np.linalg.norm(w_opt_fda)
if w_opt_fda[0] < 0:
    w_opt_fda = -w_opt_fda
theta_sub = np.radians(30)
w_sub_fda = np.array([np.cos(theta_sub), np.sin(theta_sub)])

fig_fda, (ax_fda_2d, ax_sub, ax_opt) = plt.subplots(1, 3, figsize=(15, 5))
ax_fda_2d.scatter(X_fda_1[:, 0], X_fda_1[:, 1], s=10, alpha=0.35, color="tab:blue")
ax_fda_2d.scatter(X_fda_2[:, 0], X_fda_2[:, 1], s=10, alpha=0.35, color="tab:orange")
for vec_fda, col_fda, lab_fda in [
    (w_sub_fda * 3, "k", "$w$"),
    (w_opt_fda * 3, "tab:green", r"$w_{\mathrm{opt}}$"),
]:
    ax_fda_2d.annotate("", xy=mu_fda + vec_fda, xytext=mu_fda,
                       arrowprops=dict(arrowstyle="->", lw=2.5, color=col_fda))
    ax_fda_2d.text(*(mu_fda + vec_fda * 1.08), lab_fda, color=col_fda, fontsize=13)
ax_fda_2d.set_xlabel("$x_1$")
ax_fda_2d.set_ylabel("$x_2$")
ax_fda_2d.set_aspect("equal")
ax_fda_2d.set_title("Direction $w$ vs Fisher optimum")

for ax_i, w_i, lab_i in [
    (ax_sub, w_sub_fda, "$w$"),
    (ax_opt, w_opt_fda, r"$w_{\mathrm{opt}}$"),
]:
    ax_i.hist(X_fda_1 @ w_i, bins=30, alpha=0.6, color="tab:blue")
    ax_i.hist(X_fda_2 @ w_i, bins=30, alpha=0.6, color="tab:orange")
    ax_i.set_xlabel("projected coordinate $z = \\mathbf{w}^T \\mathbf{x}$")
    ax_i.set_ylabel("count")
    ax_i.set_title(f"{lab_i}: FDR = {fdr_of(w_i, X_fda_1, X_fda_2):.2f}")

display(fig_fda)
plt.close(fig_fda)


### Remark

- Generalized to $\mathbf{z} = \mathbf{W}^T \mathbf{x} \in \mathbb{R}^k$ where $k \leq d$ and $\mathbf{W} \in \mathbb{R}^{d \times k}$.
    - More complex (pages 291-297 in book).


## Principal Component Analysis (PCA)

- First: $\mathbf{z} = \mathbf{A} \mathbf{x}$ such that $\mathbf{z} \in \mathbb{R}^d$, $\mathbf{x} \in \mathbb{R}^d$, and $\mathbf{A} \in \mathbb{R}^{d \times d}$
- Want: $\boldsymbol{\Sigma}_z$ diagonal!


### A closer look at the covariance matrix

- Have:

$$
\boldsymbol{\Sigma}_z = \mathbb{E}[(\mathbf{z} - \boldsymbol{\mu}_z)(\mathbf{z} - \boldsymbol{\mu}_z)^T]
$$

$$
= \mathbb{E}[(\mathbf{A}\mathbf{x} - \mathbf{A}\boldsymbol{\mu}_x)(\mathbf{A}\mathbf{x} - \mathbf{A}\boldsymbol{\mu}_x)^T]
= \mathbf{A} \, \boldsymbol{\Sigma}_x \mathbf{A}^T
$$

---

- $\boldsymbol{\Sigma}_x$: symmetric and positive semi-definite $\implies$ orthogonal eigenvectors and non-negative eigenvalues.


### Eigendecomposition of the covariance matrix

- Let $\mathbf{E} = [\mathbf{e}_1, \ldots, \mathbf{e}_d]$

$$
\boldsymbol{\Sigma}_x \mathbf{E} = \mathbf{E} \boldsymbol{\Lambda}
$$

where
$$
\boldsymbol{\Lambda} = \begin{bmatrix}
\lambda_1 & 0 & \cdots & 0 \\
0 & \lambda_2 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \lambda_d
\end{bmatrix}
$$
(diagonal matrix of eigenvalues)

---

- With $\mathbf{A} = \mathbf{E}^T$ (orthonormal), the covariance of the
  transformed data is diagonal:

$$
\boldsymbol{\Sigma}_z = \mathbf{E}^T \boldsymbol{\Sigma}_x \mathbf{E} = \boldsymbol{\Lambda}
$$


### Interpreting the eigenvalues and eigenvectors

- $\boldsymbol{\Sigma}_z = \boldsymbol{\Lambda}$: the transformed features
  are uncorrelated.
- The variance of $z_i$ equals $\lambda_i$.
- The eigenvectors $\mathbf{e}_i$ are the directions of maximal variance in
  the original space.


### Variance maximally preserved

- First: $\sum_{i=1}^d \text{Var}(z_i) = \operatorname{trace}(\boldsymbol{\Sigma}_z) = \sum_{i=1}^d \lambda_i$

- Thus: Let $\mathbf{A} = \mathbf{E}^T = [\mathbf{e}_1, \ldots, \mathbf{e}_d]^T$

- Remark: assume $\mathbb{E}[\mathbf{x}] = 0$ (center the data first).


### Example: projecting onto the eigenvectors

- The data cloud below is **unlabeled** — PCA only sees the point cloud.
- The left panel shows both eigenvectors of $\Sigma_x$; the other two show
  the 1-D projections onto $e_1$ (most variance) and $e_2$ (least).
- Compare the spread of the projections with $\sqrt{\lambda_i}$.


In [ ]:
# Unsupervised PCA illustration: two overlapping 2-D blobs. The labels
# used to generate the data are never seen by the eigendecomposition.
rng_pca = np.random.default_rng(0)
X_pca = np.vstack(
    [
        rng_pca.normal(loc=[-1.0, 0.3], scale=[0.55, 0.35], size=(150, 2)),
        rng_pca.normal(loc=[0.6, -0.4], scale=[0.65, 0.55], size=(150, 2)),
    ]
)
mu_pca = X_pca.mean(axis=0)
Xc_pca = X_pca - mu_pca
cov_pca = np.cov(Xc_pca, rowvar=False)
w_pca, V_pca = np.linalg.eigh(cov_pca)
order_pca = np.argsort(w_pca)[::-1]
w_pca, V_pca = w_pca[order_pca], V_pca[:, order_pca]

fig_pca, (ax_pca_2d, ax_pca_e1, ax_pca_e2) = plt.subplots(1, 3, figsize=(15, 4.8))

ax_pca_2d.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.35, s=12, color="gray")
scale_pca = 2.5 * np.sqrt(w_pca.max())
for j in range(2):
    ev = V_pca[:, j] * scale_pca
    ax_pca_2d.annotate("", xy=mu_pca + ev, xytext=mu_pca,
                       arrowprops=dict(arrowstyle="->", lw=2.5, color=f"C{j}"))
    ax_pca_2d.text(*(mu_pca + ev * 1.15), f"$e_{{{j + 1}}}$",
                   fontsize=15, ha="center", color=f"C{j}")
ax_pca_2d.set_aspect("equal")
ax_pca_2d.set_title("Eigenvectors of $\\Sigma_x$")

for ax_i, j_i in [(ax_pca_e1, 0), (ax_pca_e2, 1)]:
    proj_i = Xc_pca @ V_pca[:, j_i]
    ax_i.hist(proj_i, bins=30, color="tab:red", alpha=0.75, edgecolor="white")
    ax_i.set_xlabel(f"projection onto $e_{{{j_i + 1}}}$")
    ax_i.set_ylabel("count")
    ax_i.set_title(
        f"$\\lambda_{{{j_i + 1}}} = {w_pca[j_i]:.2f}$  "
        f"({100 * w_pca[j_i] / w_pca.sum():.0f}\\% of variance)"
    )

display(fig_pca)
plt.close(fig_pca)


## PCA is reconstruction / compression

- Let $\mathbf{z} \in \mathbb{R}^d = [z(0), z(1), \ldots, z(d-1)]^T$


### PCA is reconstruction / compression

- Have $\mathbf{x} = \mathbf{A}^T \mathbf{z}$ (with $\mathbf{A} = \mathbf{E}^T$)


### PCA is reconstruction / compression

- **MSE:** $\mathbb{E}\left[\|\mathbf{x} - \hat{\mathbf{x}}\|^2\right]$

For $\mathbf{z} \in \mathbb{R}^k$: $\hat{\mathbf{x}} = \sum_{i=0}^{k-1} z(i) \mathbf{e}_i$

If $z(i) = 0$ for $i \geq k$:
$$
\mathbb{E}\left[\|\mathbf{x} - \hat{\mathbf{x}}\|^2\right] = \sum_{i=k}^{d-1} \lambda_i
$$

$\implies$ **MSE is minimized**.

---

- **Compression:** Store/save $\mathbf{z} \in \mathbb{R}^k$ instead of $\mathbf{x}$ (e.g. images).
- **Reconstruct:** $\hat{\mathbf{x}}$ using $\mathbf{z}$.


### Example: compressing handwritten digits

- Each $8 \times 8$ digit image is a point in $\mathbb{R}^{64}$.
- Top row: the same digit reconstructed from the first $k$ principal
  components.
- Bottom: the reconstruction error follows $\sum_{i \geq k} \lambda_i$.


In [ ]:
# PCA reconstruction on the digits dataset (bundled with scikit-learn,
# so no network access). The empirical MSE is compared with the
# theoretical tail sum of the eigenvalues from the slides.
from sklearn.datasets import load_digits

dgt_rec = load_digits()
X_rec = dgt_rec.data.astype(float)
mu_rec = X_rec.mean(axis=0)
Xc_rec = X_rec - mu_rec
_, s_rec, Vt_rec = np.linalg.svd(Xc_rec, full_matrices=False)
lam_rec = s_rec**2 / (len(X_rec) - 1)  # eigenvalues of the covariance

ks_rec = np.arange(1, len(lam_rec) + 1)
mse_curve = np.array(
    [
        ((Xc_rec - (Xc_rec @ Vt_rec[:k].T) @ Vt_rec[:k]) ** 2).sum(axis=1).mean()
        for k in ks_rec
    ]
)
# Keeping k components leaves eigenvalues i = k, ..., d-1.
theory_curve = np.array([lam_rec[k:].sum() for k in ks_rec])

fig_rec = plt.figure(figsize=(16, 5))
outer_rec = fig_rec.add_gridspec(1, 2, width_ratios=(1, 1.5), wspace=0.25)

ax_2d_rec = fig_rec.add_subplot(outer_rec[0, 0])
Z2_rec = Xc_rec @ Vt_rec[:2].T
ax_2d_rec.scatter(Z2_rec[:, 0], Z2_rec[:, 1], c=dgt_rec.target, cmap="tab10", s=8, alpha=0.6)
ax_2d_rec.set_xlabel("$z_1$ (1st PC)")
ax_2d_rec.set_ylabel("$z_2$ (2nd PC)")
ax_2d_rec.set_title("Digits in 2-D (first two PCs)")

inner_rec = outer_rec[0, 1].subgridspec(2, 4, height_ratios=(1, 1.4), hspace=0.35, wspace=0.15)
for col_rec, k_rec_v in enumerate([1, 5, 15, 40]):
    z_rec = Xc_rec @ Vt_rec[:k_rec_v].T
    Xhat_rec = z_rec @ Vt_rec[:k_rec_v] + mu_rec
    ax_rec = fig_rec.add_subplot(inner_rec[0, col_rec])
    ax_rec.imshow(Xhat_rec[0].reshape(8, 8), cmap="gray_r")
    ax_rec.set_title(f"$k={k_rec_v}$")
    ax_rec.axis("off")

ax_err = fig_rec.add_subplot(inner_rec[1, :])
ax_err.plot(ks_rec, mse_curve, label="empirical MSE")
ax_err.plot(ks_rec, theory_curve, "--", label=r"$\sum_{i \geq k} \lambda_i$")
ax_err.set_xlabel("components kept, $k$")
ax_err.set_ylabel("reconstruction MSE")
ax_err.legend()

display(fig_rec)
plt.close(fig_rec)


## Programming exercises

Below are programming exercises associated with this lecture. These cell blocks are starting points that load the data and prepare the problem such that you can get going with the implementation. There are also theoretical exercises, but due to copyright we cannot share them here. They will be made available in a private repository connected to the course.


### Dimensionality reduction on the Iris dataset

We will now revisit the Iris dataset from the first lectures. Instead of simply selecting 2 features, you will now perform either FDA or PCA to reduce the dimensionality down to 2 and visualize the data. Do you observe any difference between the two methods?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

# fetch dataset
iris = fetch_ucirepo(id=53)

# data (as pandas dataframes)
X = iris.data.features
feature_1_name = 'sepal length (cm)'
feature_2_name = 'sepal width (cm)'
y = np.zeros(150)
y[50:100] = 1
y[100:150] = 2
y_names = np.unique(iris.data.targets)

